# Two-Tunable Qubit Hamiltonian

Model and plumbing live in `two_qubit_model.py`; this notebook sets parameters, builds control pulses and rolls out the dynamics.

**Frame:** rotating at each qubit's idle frequency, RWA.

$$\frac{H(t)}{2\pi} = \sum_i \Big[\Delta_i(t)\, n_i + \tfrac{\alpha_i}{2} b_i^{\dagger 2} b_i^2 + \tfrac{u_{x,i}}{2}(b_i + b_i^\dagger) + \tfrac{u_{y,i}}{2}\, i(b_i^\dagger - b_i)\Big] + J\big(e^{i\delta t} b_1^\dagger b_2 + \text{h.c.}\big)$$

* $\Delta_i(t) = f_i(I(t)) - f_{\text{idle},i}$, with $f_i(\Phi) = f_{\max,i}\sqrt{|\cos(\pi\Phi_i/\Phi_0)|}$ and $\Phi_i = \sum_j M_{ij} I_j$ (crosstalk included).
* $\delta = 2\pi(f_{\text{idle},1} - f_{\text{idle},2})$: the coupling only acts efficiently when flux brings the qubits into resonance.
* `n_levels=2` gives the Pauli model; `n_levels=3` adds leakage.

**Units:** ns, GHz (cyclic), mA, pH. **Controls** are piecewise constant on `N` slices of width `dt`: `I`, `ux`, `uy` each have shape `(2, N)`.

In [ ]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

from two_qubit_model import QubitParams, SystemParams, Controls, TwoQubitModel

## System parameters

In [ ]:
params = SystemParams(
    q1=QubitParams(f_max=6.0, alpha=-0.20),   # GHz
    q2=QubitParams(f_max=5.6, alpha=-0.21),
    J=0.010,                                  # GHz (10 MHz)
    M=[[2.00, 0.05],                          # pH, M[i, j]: flux line j -> SQUID i
       [0.05, 2.00]],
    I_idle=[0.0, 0.0],                        # mA (both parked at the sweet spot)
    n_levels=3,
)
model = TwoQubitModel(params)
print("idle frequencies [GHz]:", model.f_idle)
print("idle detuning delta/2pi [GHz]:", model.delta / (2 * np.pi))

## Example control pulses
A Gaussian X π pulse on qubit 1 (|00⟩ → |10⟩), then a flux pulse that tunes qubit 1 into resonance with qubit 2 for $1/(4J)$ (|10⟩ → |01⟩).
Replace these arrays with your optimizer's output.

In [ ]:
dt = 0.02                                  # ns per slice
t_pi, t_swap = 20.0, 1 / (4 * params.J)    # ns
N_pi, N_swap = round(t_pi / dt), round(t_swap / dt)
N = N_pi + N_swap

ctrl = Controls.idle(model, N, dt)          # flux at idle, drives off

# Gaussian pi pulse on qubit 1, normalized so the area gives exactly a pi rotation
t = ctrl.t_mid[:N_pi]
g = np.exp(-0.5 * ((t - t_pi / 2) / (t_pi / 5)) ** 2)
ctrl.ux[0, :N_pi] = 0.5 * g / (g.sum() * dt)

# Flux pulse: currents that put qubit 1 at qubit 2's idle frequency (crosstalk-corrected)
I_res = model.currents_for_frequencies([model.f_idle[1], model.f_idle[1]])
ctrl.I[:, N_pi:] = I_res[:, None]
print("resonance currents [mA]:", I_res)

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(8, 4))
for i in range(2):
    ax[0].step(ctrl.t_mid, ctrl.ux[i], where="mid", label=f"$u_{{x,{i+1}}}$")
    ax[1].step(ctrl.t_mid, ctrl.I[i], where="mid", label=f"$I_{i+1}$")
ax[0].set_ylabel("drive [GHz]"); ax[1].set_ylabel("current [mA]"); ax[1].set_xlabel("t [ns]")
ax[0].legend(); ax[1].legend(); plt.tight_layout()

## Rollout

In [ ]:
labels = ["00", "01", "10", "11", "leak"]
P = [model.basis(0, 0).proj(), model.basis(0, 1).proj(), model.basis(1, 0).proj(), model.basis(1, 1).proj()]
P.append(qt.qeye(model.dims) - sum(P))

# Closed system. Pass c_ops=model.collapse_ops(T1=(30e3, 30e3), Tphi=(40e3, 40e3)) for decoherence.
res = model.evolve(ctrl, model.basis(0, 0), e_ops=P)

plt.figure(figsize=(8, 3))
for lab, p in zip(labels, res.expect):
    plt.plot(res.times, p, label=lab)
plt.xlabel("t [ns]"); plt.ylabel("population"); plt.legend(); plt.tight_layout()
print({lab: round(float(p[-1]), 5) for lab, p in zip(labels, res.expect)})

## GRAPE hooks
* `model.slice_hamiltonians(ctrl)`: dense $H_k$ per slice, shape `(N, D, D)`
* `model.slice_propagators(ctrl)`: $U_k = e^{-iH_k\,dt}$
* `model.propagator(ctrl, "pwc" | "ode")`: total unitary
* `model.control_operators()`: $\partial H/\partial u$ for the z, x, y controls
* `model.detuning_jacobian(I)`: $\partial\Delta_i/\partial I_j$, the chain rule for optimizing in current space (it is zero at a sweet spot)
* `model.gate_fidelity(U, target)`: leakage-aware average gate fidelity on the computational subspace

The pack/unpack helpers below give a flat parameter vector for an external optimizer.

In [ ]:
def pack(ctrl: Controls) -> np.ndarray:
    return np.concatenate([ctrl.I.ravel(), ctrl.ux.ravel(), ctrl.uy.ravel()])

def unpack(x: np.ndarray, n_slices: int, dt: float) -> Controls:
    I, ux, uy = np.split(np.asarray(x), 3)
    return Controls(dt, I.reshape(2, n_slices), ux.reshape(2, n_slices), uy.reshape(2, n_slices))

def infidelity(x, target, n_slices, dt):
    return 1 - model.gate_fidelity(model.propagator(unpack(x, n_slices, dt)), target)

# Example: score just the pi-pulse segment against X on qubit 1
X1 = qt.tensor(qt.sigmax(), qt.qeye(2)).full()
x_pi = pack(Controls(dt, ctrl.I[:, :N_pi], ctrl.ux[:, :N_pi], ctrl.uy[:, :N_pi]))
print("infidelity (pi pulse vs X1):", infidelity(x_pi, X1, N_pi, dt))

# Full sequence: magnitude of the computational-subspace block
U = model.propagator(ctrl, "pwc")
print("|U_sub| (rows/cols 00, 01, 10, 11):\n", np.round(np.abs(model.project_computational(U)), 3))